In [1]:
import os
import requests
from typing import List, Dict, Any, TypedDict
from langchain_community.document_loaders import TextLoader

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Weaviate
from langchain_openai import ChatOpenAI
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema.runnable import RunnablePassthrough
from langgraph.graph import StateGraph, END
import weaviate
from weaviate.embedded import EmbeddedOptions
import dotenv

# Load environment variables (e.g., OPENAI_API_KEY)
dotenv.load_dotenv()
# Set your OpenAI API key (ensure it's loaded from .env or set here)
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# --- 1. Data Preparation (Preprocessing) ---
# Load data
url = "https://frontiernerds.com/files/state_of_the_union.txt"
res = requests.get(url)

with open("state_of_the_union.txt", "w") as f:
    f.write(res.text)

loader = TextLoader('./state_of_the_union.txt')
documents = loader.load()

# Chunk documents
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

# Embed and store chunks in Weaviate
client = weaviate.Client(
    embedded_options = EmbeddedOptions()
)

vectorstore = Weaviate.from_documents(
    client = client,
    documents = chunks,
    embedding = OpenAIEmbeddings(),
    by_text = False
)

# Define the retriever
retriever = vectorstore.as_retriever()

# Initialize LLM
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

# --- 2. Define the State for LangGraph ---
class RAGGraphState(TypedDict):
    question: str
    documents: List[Document]
    generation: str

# --- 3. Define the Nodes (Functions) ---

def retrieve_documents_node(state: RAGGraphState) -> RAGGraphState:
    """Retrieves documents based on the user's question."""
    question = state["question"]
    documents = retriever.invoke(question)
    return {"documents": documents, "question": question, "generation": ""}

def generate_response_node(state: RAGGraphState) -> RAGGraphState:
    """Generates a response using the LLM based on retrieved documents."""
    question = state["question"]
    documents = state["documents"]

    # Prompt template from the PDF
    template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""
    prompt = ChatPromptTemplate.from_template(template)

    # Format the context from the documents
    context = "\n\n".join([doc.page_content for doc in documents])

    # Create the RAG chain
    rag_chain = prompt | llm | StrOutputParser()

    # Invoke the chain
    generation = rag_chain.invoke({"context": context, "question": question})
    return {"question": question, "documents": documents, "generation": generation}

# --- 4. Build the LangGraph Graph ---

workflow = StateGraph(RAGGraphState)

# Add nodes
workflow.add_node("retrieve", retrieve_documents_node)
workflow.add_node("generate", generate_response_node)

# Set the entry point
workflow.set_entry_point("retrieve")

# Add edges (transitions)
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

# Compile the graph
app = workflow.compile()

# --- 5. Run the RAG Application ---
if __name__ == "__main__":
    print("\n--- Running RAG Query ---")
    query = "What did the president say about Justice Breyer"
    inputs = {"question": query}
    for s in app.stream(inputs):
        print(s)

    print("\n--- Running another RAG Query ---")
    query_2 = "What did the president say about the economy?"
    inputs_2 = {"question": query_2}
    for s in app.stream(inputs_2):
        print(s)

D:\Anaconda\envs\Agentic-Design-Patterns\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
Created a chunk of size 642, which is longer than the specified 500
Created a chunk of size 514, which is longer than the specified 500
Created a chunk of size 503, which is longer than the specified 500
Created a chunk of size 528, which is longer than the specified 500
Created a chunk of size 683, which is longer than the specified 500
Created a chunk of size 585, which is longer than the specified 500
Created a chunk of size 529, which is longer than the specified 500
Created a chunk of size 614, which is longer than the specified 500
Created a chunk of size 512, which is longer than the specified 500
Create


--- Running RAG Query ---


D:\Anaconda\envs\Agentic-Design-Patterns\Lib\site-packages\langchain_community\embeddings\openai.py:503: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  response = response.dict()


{'retrieve': {'documents': [Document(metadata={'source': './state_of_the_union.txt'}, page_content="We must continually renew this promise. My administration has a civil rights division that is once again prosecuting civil rights violations and employment discrimination. We finally strengthened our laws to protect against crimes driven by hate. This year, I will work with Congress and our military to finally repeal the law that denies gay Americans the right to serve the country they love because of who they are. We are going to crack down on violations of equal pay laws -- so that women get equal pay for an equal day's work. And we should continue the work of fixing our broken immigration system -- to secure our borders, enforce our laws and ensure that everyone who plays by the rules can contribute to our economy and enrich our nations."), Document(metadata={'source': './state_of_the_union.txt'}, page_content='Madame Speaker, Vice President Biden, members of Congress, distinguished g

D:\Anaconda\envs\Agentic-Design-Patterns\Lib\site-packages\langchain_community\embeddings\openai.py:503: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  response = response.dict()


{'retrieve': {'documents': [Document(metadata={'source': './state_of_the_union.txt'}, page_content="But when I ran for president, I promised I wouldn't just do what was popular -- I would do what was necessary. And if we had allowed the meltdown of the financial system, unemployment might be double what it is today. More businesses would certainly have closed. More homes would have surely been lost."), Document(metadata={'source': './state_of_the_union.txt'}, page_content="It begins with our economy.\n\nOur most urgent task upon taking office was to shore up the same banks that helped cause this crisis. It was not easy to do. And if there's one thing that has unified Democrats and Republicans, it's that we all hated the bank bailout. I hated it. You hated it. It was about as popular as a root canal."), Document(metadata={'source': './state_of_the_union.txt'}, page_content='We cannot afford another so-called economic expansion like the one from last decade -- what some call the lost dec